# 12.e — Ad hoc test: the suitable area weighted by the probability (continuous) against the binary area

**Ad hoc test, outside the suitability chain**: it reads the BIO variables already written by 1.1.d, 12.a, 12.c and
12.d (nothing is recomputed from the daily climate), the soil of 2.g and the SVM of 3.g, and writes only in
`<out_root>/test_continuous_area/<route>/<set>/`.

**Question.** In 6.b a cell is suitable or not: probability ≥ 0.5 (and not novel), ECOCROP TMIN, soil ≥ 10, and the
areas count each suitable cell in full. A cell at p = 0.49 counts 0, at 0.51 its whole area: where many cells of a crop
lie near the threshold, a small shift of the climate moves the area by steps. 12.d found the effect of the AMOC
weakening on today's cropland of spring wheat between +15 and +21 points depending on the ERA5 reference (soybean +1.3
to +3.9). Is this the threshold, and does a continuous measure make the effect more stable?

**Method.**
- *Binary* (= 6.b): area of the suitable cells.
- *Continuous*: each cell counts for its probability at equal priors (0 where the climate is novel), on the cells that
  pass the fixed floors (ECOCROP TMIN of the lower part of the cell, soil ≥ 10): Σ area × p × floor. The floors stay
  binary (physiological and soil limits); only the SVM probability becomes a weight. The probability is not the share of
  the cell that is suitable: the continuous area is a sensitivity measure, not a replacement of the maps.
- Same metrics as 6.b / 12.a / 12.d: today's crop area (points of the MIRCA2000 area of the crop) and suitable land per
  region (%; cells assessable in both runs). For both measures: the effect of the AMOC weakening on the three ERA5
  references of 12.d (spread between them), and the noise of the runs from their two halves (12.a: SE of the full effect
  = √(mean d² / 2), d = H1 − H2 of the four runs).
- *Near the threshold*: share of today's crop area with 0.4 ≤ p < 0.6 (floors passed), on ERA5 and in the 2050 runs.

**Reading.** If the continuous effect has the same sign and size as the binary one, with a smaller spread between the
references and a smaller SE, the sensitivity of 12.d is the step of the threshold and the binary effect is a noisier
estimate of the same signal; the continuous values then give the stable size.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore'); os.environ["PYTHONWARNINGS"] = "ignore"   # also in the joblib workers

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = "no_rad"
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
NOISE_RUNS = RUNS                 # runs whose two halves (12.a) give the noise
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
REFS = ["full", "H1", "H2"]
LEVELS = ["low", "high"]
BAND = (0.4, 0.6)                 # "near the threshold"
KERNEL_EPS = 0.01
N_JOBS = min(5, os.cpu_count() or 1)
assert CHOSEN_RAD == "no_rad", "no_rad only (no rsds in the runs)"

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
BIO12A = cu.out_dir(CFG, "test_internal_variability", ROUTE, SET) / "bio"    # 12.a: delta BIO of the halves of the runs
BIO12C = cu.out_dir(CFG, "test_era5_robustness", ROUTE, SET) / "bio"        # 12.c: BIO of the halves of ERA5
BIO12D = cu.out_dir(CFG, "test_era5_reference", ROUTE, SET) / "bio"         # 12.d: delta BIO on the three references
OUT = cu.out_dir(CFG, "test_continuous_area", ROUTE, SET)                   # the only folder written

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_MHA = np.nan_to_num(cu.land_area_ha(CFG)) * LAND / 1e6
DT = cu.tmin_warming(CFG)
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
era5_file = lambda crop: ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
with xr.open_dataset(era5_file(CROPS[0])) as d0:
    COLS = [v for v in d0.data_vars if v.startswith("BIO") and not v.endswith("e") and v not in ("BIO03", "BIO08")]


def files_of(crop):
    # every climate of the test: (reference, "reference" or run) of 12.c / 12.d, ("half", run, part) of 12.a
    f = {("full", "reference"): era5_file(crop),
         **{(r, "reference"): BIO12C / f"BIOvariables_ERA5_{r}_{crop}.nc" for r in REFS if r != "full"},
         **{(r, run): BIO12D / f"BIOvariables_delta_{r}_{run}_{crop}.nc" for r in REFS for run in RUNS},
         **{("half", run, p): BIO12A / f"BIOvariables_delta_{run}_{p}_{crop}.nc" for run in NOISE_RUNS for p in ("H1", "H2")}}
    miss = [str(v) for v in f.values() if not v.exists()]
    assert not miss, f"missing (run 12.a, 12.c and 12.d first): {miss[:3]}"
    return f


print(f"set {SET} | references {REFS} | runs {RUNS} | noise from the halves of {NOISE_RUNS} | written only in {OUT}")

## 1. Probability and binary suitability of every climate (function of 12.a, with the probability)

In [ ]:
def suitability(crop, files):
    # {key of files: {"suit": {level: bool}, "novel": bool, "p": probability at equal priors, 0 where novel,
    #  "floor": {level: bool, ECOCROP TMIN and soil >= 10}}}: the function of 12.a, with p and the fixed floors
    bundle = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    cols, n1, n0 = bundle["bio_cols"], bundle["n_class1"], bundle["n_class0"]
    assert set(cols) <= set(COLS), (cols, COLS)
    tr = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    ii = np.searchsorted(GRID["lat"], tr.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], tr.lon.values - 1e-6)
    with xr.open_dataset(era5_file(crop)) as ds:
        REF = cu.bio_array(ds, cols, CFG)[ii, jj]
    REF = REF[np.isfinite(REF).all(axis=1)]
    kfun = cu.kernel_support(bundle["model"])
    Fd = cu.crossfit_models(MODELS / f"folds_{TAG}_{crop}.joblib", tr[cols].values.astype(float),
                            tr["label"].values.astype(int), bundle, n_jobs=N_JOBS)
    ssr = {lv: np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0) for lv in LEVELS}
    which = cu.ECOCROP_TMIN[crop][1]
    out = {}
    for key, path in files.items():
        with xr.open_dataset(path) as ds:
            A = cu.bio_array(ds, cols, CFG)
            tg = np.asarray(cu.growing_temperature(ds, which), float)
        X = A.reshape(-1, len(cols))
        ok = np.isfinite(X).all(axis=1) & LAND.ravel()
        p, nov = np.full(X.shape[0], np.nan), np.zeros(X.shape[0], bool)
        if ok.any():                                              # (a part without any season: all NaN)
            Xs = bundle["scaler"].transform(X[ok])
            p[ok] = bundle["model"].predict_proba(Xs)[:, 1]
            nov[ok] = (cu.mess(X[ok], REF)[0] < 0) | (cu.nt2(X[ok], REF) > 1) | (kfun(Xs) < KERNEL_EPS)
        p, nov = p.reshape(LAND.shape), nov.reshape(LAND.shape)
        g = cu.crossfit_proba(Fd, A[ii, jj])                      # training cells: the fold model that did not see them
        fin = np.isfinite(g)
        p[ii[fin], jj[fin]] = g[fin]
        p = cu.balanced_proba(p, n1, n0)
        nov &= np.isfinite(p)
        proba = np.where(nov, 0.0, p)
        clim = cu.climate_suitable(proba, tg, crop, DT) & np.isfinite(proba)
        tmin_ok = cu.climate_suitable(np.ones(LAND.shape), tg, crop, DT)           # the ECOCROP TMIN alone
        out[key] = {"suit": {lv: LAND & cu.suitable(clim, ssr[lv]) for lv in LEVELS}, "novel": nov,
                    "p": np.where(LAND & np.isfinite(proba), proba, 0.0),
                    "floor": {lv: LAND & tmin_ok & cu.suitable(np.ones(LAND.shape), ssr[lv]) for lv in LEVELS}}
    return out


mha = lambda w: float(np.nansum(LAND_MHA * w))
pct = lambda a, b: 100 * (a - b) / b if b > 0.5 else np.nan        # < 0.5 Mha: not reported (as 6.b)
METRICS = ["today's crop area (points)", *[f"{r} land (%)" for r in REGIONS]]


def weights(c, lv, measure):
    # area weight of every cell: binary (suitable 1 / 0) or continuous (p x floors)
    return c["suit"][lv].astype(float) if measure == "binary" else c["p"] * c["floor"][lv]


def change(A, B, lv, measure, crop_ha):
    # metrics of the change from climate B to climate A (as 6.b / 12.a): today's crop area (points), land per region (%)
    wa, wb = weights(A, lv, measure), weights(B, lv, measure)
    both = LAND & ~(A["novel"] | B["novel"])
    m = {"today's crop area (points)": 100 * (crop_ha * (wa - wb)).sum() / crop_ha.sum()}
    for reg, rm in REGIONS.items():
        m[f"{reg} land (%)"] = pct(mha(wa * (both & rm)), mha(wb * (both & rm)))
    return m


rows, noise, band, check = [], [], [], []
for crop in CROPS:
    t0 = time.time()
    R = suitability(crop, files_of(crop))
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    for key, c in R.items():                   # check: binary = p >= 0.5 and floors
        for lv in LEVELS:
            check.append({"crop": crop, "level": lv, "cells differing": int(((c["p"] >= cu.CLIMATE_P) & c["floor"][lv] != c["suit"][lv])[LAND].sum())})
    for lv in LEVELS:
        for measure in ("binary", "continuous"):
            for ref in REFS:
                for h, o in PAIRS:
                    for k, v in change(R[(ref, h)], R[(ref, o)], lv, measure, crop_ha).items():
                        rows.append({"crop": crop, "level": lv, "measure": measure, "pair": f"{h} - {o}", "metric": k,
                                     "reference": ref, "effect": v})
            for run in NOISE_RUNS:
                for k, v in change(R[("half", run, "H1")], R[("half", run, "H2")], lv, measure, crop_ha).items():
                    noise.append({"crop": crop, "level": lv, "measure": measure, "run": run, "metric": k, "d (H1 - H2)": v})
        for key in [("full", "reference"), *[("full", r) for r in RUNS]]:
            c = R[key]
            near = (c["p"] >= BAND[0]) & (c["p"] < BAND[1]) & c["floor"][lv]
            band.append({"crop": crop, "level": lv, "climate": key[1],
                         f"today's crop area with {BAND[0]} <= p < {BAND[1]} (%)": 100 * crop_ha[near].sum() / crop_ha.sum(),
                         "today's crop area suitable, binary (%)": 100 * crop_ha[c["suit"][lv]].sum() / crop_ha.sum(),
                         "today's crop area, continuous (%)": 100 * (crop_ha * c["p"] * c["floor"][lv]).sum() / crop_ha.sum()})
    del R
    print(f"{crop}: {(time.time() - t0) / 60:.1f} min", flush=True)
CHECK = pd.DataFrame(check)
print("check, binary = (p >= 0.5) and floors: largest number of differing cells", int(CHECK["cells differing"].max()))

## 2. Binary against continuous: effect, spread over the references, noise of the runs

`spread` = largest − smallest effect over the three references; `SE runs` = √(mean d² / 2) of the halves of the runs;
`|full| / SE`: the effect against the noise; `spread / SE`: the dependence on the reference against the noise.

In [ ]:
EFF = pd.DataFrame(rows).pivot_table(index=["crop", "level", "measure", "pair", "metric"], columns="reference",
                                     values="effect", sort=False)[REFS]
EFF["spread"] = EFF.max(axis=1) - EFF.min(axis=1)
EFF["same sign"] = (np.sign(EFF[REFS]).nunique(axis=1) == 1) & (EFF[REFS].abs().min(axis=1) > 0)
NOISE = pd.DataFrame(noise)
SE = NOISE.groupby(["crop", "level", "measure", "metric"], sort=False)["d (H1 - H2)"].apply(lambda d: np.sqrt(np.nanmean(d ** 2) / 2))
EFF = EFF.join(SE.rename("SE runs"), on=["crop", "level", "measure", "metric"])
EFF["|full| / SE"] = EFF["full"].abs() / EFF["SE runs"]
EFF["spread / SE"] = EFF["spread"] / EFF["SE runs"]
BAND_T = pd.DataFrame(band).set_index(["crop", "level", "climate"])
EFF.to_csv(OUT / "effect_binary_vs_continuous.csv"); NOISE.to_csv(OUT / "noise_halves.csv", index=False)
BAND_T.to_csv(OUT / "near_threshold.csv")
with pd.option_context("display.max_rows", 500, "display.width", 250):
    for met in ["today's crop area (points)", "global land (%)", "north of 50 N land (%)"]:
        for lv in LEVELS[::-1]:
            t = EFF.xs((lv, met), level=("level", "metric")).reset_index()
            t = t[t.pair == f"{PAIRS[-1][0]} - {PAIRS[-1][1]}"].drop(columns="pair")
            print(f"{met}, {lv} input, {PAIRS[-1][0]} - {PAIRS[-1][1]}: binary against continuous")
            display(t.set_index(["crop", "measure"]).round(2))
    print("Near the threshold (high input)")
    display(BAND_T.xs("high", level="level").round(1))

## 3. Figure: effect of the AMOC weakening in 2050 on today's cropland, binary and continuous, three references

In [ ]:
LV = "high" if "high" in LEVELS else LEVELS[0]
h, o = PAIRS[-1]
mets = ["today's crop area (points)", "global land (%)", "north of 50 N land (%)"]
cols = {"full": "#4d4d4d", "H1": "#2166ac", "H2": "#b2182b"}
fig, axes = plt.subplots(1, len(mets), figsize=(6.5 * len(mets), 0.75 * len(CROPS) + 2), squeeze=False)
for ax, met in zip(axes[0], mets):
    labels, y = [], 0
    for crop in CROPS:
        for measure in ("binary", "continuous"):
            key = (crop, LV, measure, f"{h} - {o}", met)
            if key not in EFF.index:
                continue
            r = EFF.loc[key]
            for n, ref in enumerate(REFS):
                ax.barh(y + (n - 1) * 0.27, r[ref], height=0.25, color=cols[ref], alpha=1.0 if measure == "binary" else 0.55,
                        hatch=None if measure == "binary" else "//", label=f"{ref}" if y == 0 else None)
            if np.isfinite(r["SE runs"]):
                ax.errorbar(r["full"], y - 0.27, xerr=2 * r["SE runs"], color="k", capsize=3, lw=1)
            labels.append(f"{crop.replace('_rf', '')} {measure}"); y += 1
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3); ax.set_title(met, fontsize=10)
axes[0, 0].legend(fontsize=8, title="ERA5 reference")
fig.suptitle(f"Effect of the AMOC weakening, 2050 ({LV} input): binary area (6.b) and area weighted by the probability "
             f"(hatched), three ERA5 references; black: ± 2 SE of the runs", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"binary_vs_continuous_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))